In [2]:
# ── Cell 1: Build both channels, measure truth-coverage before integrating ─
import sys, json, time
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
sys.path.insert(0, str(PROJECT_ROOT / "app"))

from corrector import SpellCorrector
sc = SpellCorrector(DATA_DIR)

# ── Channel definitions (first principles) ──
_SOUNDEX = {**{c: "1" for c in "bfpv"}, **{c: "2" for c in "cgjkqsxz"},
            **{c: "3" for c in "dt"}, "l": "4", **{c: "5" for c in "mn"}, "r": "6"}

def soundex(w):
    """Classic Soundex: first letter + 3 digits, adjacent duplicates merged."""
    w = [c for c in w.lower() if c.isalpha()]
    if not w: return ""
    first = w[0]
    codes, prev = [], _SOUNDEX.get(first, "")
    for c in w[1:]:
        d = _SOUNDEX.get(c, "")
        if c in "hw": continue
        if d and d != prev: codes.append(d)
        prev = d
    return (first.upper() + "".join(codes) + "000")[:4]

VOWELS = set("aeiou")
def skeleton(w):
    """Consonant skeleton: the word with vowels removed (texting compression)."""
    s = "".join(c for c in w if c not in VOWELS)
    return s if s else w

# ── Build the indexes over the whole dictionary ──
t0 = time.time()
sound_index, skel_index = defaultdict(list), defaultdict(list)
for w in sc.dictionary:
    sound_index[soundex(w)].append(w)
    skel_index[skeleton(w)].append(w)
print(f"Indexes built in {time.time()-t0:.1f}s — "
      f"{len(sound_index):,} soundex codes, {len(skel_index):,} skeletons")
big = max(sound_index.values(), key=len)
print(f"Largest soundex bucket: {len(big):,} words (code {soundex(big[0])})")

# ── Probe table: is the truth reachable per channel? ──
probes = [("mantainance","maintenance"), ("unexepted","unexpected"),
          ("sucraty","security"), ("hv","have"), ("sh","she"), ("ws","was"),
          ("yer","year"), ("lts","last"), ("univrsty","university"),
          ("labratary","laboratory"), ("concludsion","conclusion"),
          # from the dialect sentence:
          ("medya","media"), ("depatment","department"), ("stratgy","strategy"),
          ("moprning","morning"), ("meting","meeting"), ("confuse","confused")]
print(f"\n{'typed':<13}{'truth':<13}{'edit≤2':<8}{'soundex':<10}{'skeleton':<10}"
      f"{'snd bucket':<12}skel bucket")
for typed, truth in probes:
    pool2, _ = sc.candidates(typed)
    in_e2   = truth in pool2
    in_snd  = truth in sound_index.get(soundex(typed), [])
    in_skel = truth in skel_index.get(skeleton(typed), [])
    print(f"{typed:<13}{truth:<13}{str(in_e2):<8}{str(in_snd):<10}{str(in_skel):<10}"
          f"{len(sound_index.get(soundex(typed), [])):<12,}"
          f"{len(skel_index.get(skeleton(typed), [])):,}")

# ── Diagnostic: why didn't 'meting' and 'de' get hearings? ──
trust = json.loads((DATA_DIR / "word_trust.json").read_text())
for w in ["meting", "de", "dis", "dat"]:
    if w in sc.word_freq:
        print(f"  {w:<8} trusted (corpus veto) — corpus count {sc.word_freq[w]}")
    elif w in trust["suspicious"]:
        print(f"  {w:<8} suspicious — alts {[a for a,_ in trust['suspicious'][w]][:3]}")
    else:
        print(f"  {w:<8} quiet-rare (no strong shadow passed the gap test)")

# ── The real metric: the benchmark's unreachable pairs ──
pairs = json.loads((DATA_DIR / "misspelling_pairs.json").read_text())
print("\nScanning benchmark for tier \u22121 (no-candidate) pairs \u2014 ~2-3 min\u2026")
t0 = time.time()
unreachable = []
for i, (miss, corr) in enumerate(pairs.items()):
    if i % 1000 == 0: print(f"  {i:,}/{len(pairs):,}  ({time.time()-t0:.0f}s)")
    cands, tier = sc.candidates(miss)
    truths = corr if isinstance(corr, list) else [corr]
    if tier == -1 or not any(t in cands for t in truths):
        unreachable.append((miss, truths))

rescued_snd = sum(1 for m, ts in unreachable
                  if any(t in sound_index.get(soundex(m), []) for t in ts))
rescued_skel = sum(1 for m, ts in unreachable
                   if any(t in skel_index.get(skeleton(m), []) for t in ts))
rescued_any = sum(1 for m, ts in unreachable
                  if any(t in sound_index.get(soundex(m), []) for t in ts)
                  or any(t in skel_index.get(skeleton(m), []) for t in ts))
print(f"\nBenchmark pairs where edit\u22642 cannot reach the truth: {len(unreachable)}")
print(f"  rescued by soundex:   {rescued_snd}")
print(f"  rescued by skeleton:  {rescued_skel}")
print(f"  rescued by either:    {rescued_any}  "
      f"({rescued_any/max(1,len(unreachable)):.0%} of the unreachable class)")
print("  still unreachable:", [m for m, ts in unreachable
      if not any(t in sound_index.get(soundex(m), []) for t in ts)
      and not any(t in skel_index.get(skeleton(m), []) for t in ts)][:12])

Indexes built in 0.7s — 5,970 soundex codes, 252,781 skeletons
Largest soundex bucket: 2,470 words (code I536)

typed        truth        edit≤2  soundex   skeleton  snd bucket  skel bucket
mantainance  maintenance  False   True      True      226         1
unexepted    unexpected   False   True      False     1,007       1
sucraty      security     False   True      True      123         1
hv           have         False   True      True      76          13
sh           she          False   True      True      120         27
ws           was          False   True      True      166         20
yer          year         False   True      True      19          21
lts          last         False   False     False     113         34
univrsty     university   True    True      True      1,879       1
labratary    laboratory   True    True      True      158         5
concludsion  conclusion   True    True      False     389         0
medya        media        True    True      False     145

In [3]:
# ── Cell 2: Merge channels into the ranker, re-measure the benchmark ──────
import importlib, corrector
from edit_distance import weighted_edit_distance
importlib.reload(corrector)

class SpellCorrectorV3(corrector.SpellCorrector):
    """Phase 1: candidate pool = edit-distance ∪ soundex ∪ skeleton,
    one unified noisy-channel ranking. Indexes built from the dictionary
    at load — change the dictionary, the channels follow."""
    SND_LEN_WINDOW = 2      # soundex candidates within ±2 chars of the typed length
    SND_CAP        = 200    # hard cap per lookup, deterministic order

    def __init__(self, data_dir, **kw):
        super().__init__(data_dir, **kw)
        self.sound_index, self.skel_index = defaultdict(list), defaultdict(list)
        for w in self.dictionary:
            self.sound_index[soundex(w)].append(w)
            self.skel_index[skeleton(w)].append(w)

    def _channel_candidates(self, word):
        skel = set(self.skel_index.get(skeleton(word), []))
        bucket = self.sound_index.get(soundex(word), [])
        snd = sorted((w for w in bucket
                      if abs(len(w) - len(word)) <= self.SND_LEN_WINDOW),
                     key=lambda w: (abs(len(w) - len(word)), w))[:self.SND_CAP]
        return (skel | set(snd)) - {word}

    def suggest(self, word, left=None, right=None, top=5):
        if word in self.dictionary:
            return [(word, 0.0)], 0
        c1 = self._known(self._edits1(word))
        c2 = self._known({e2 for e1 in self._edits1(word)
                          for e2 in self._edits1(e1)}) - c1 if len(c1) < top else set()
        ch = self._channel_candidates(word) - c1 - c2
        pool = c1 | c2 | ch
        if not pool:
            return [], -1
        ranked = sorted(((w, -self.lam * weighted_edit_distance(word, w)
                          + self._ctx_logp_blend(w, left, right)) for w in pool),
                        key=lambda x: (-x[1], x[0]))
        tier = 1 if c1 else (2 if c2 else 3)
        return ranked[:top], tier

v3 = SpellCorrectorV3(DATA_DIR)

# ── The cases that started all this ──
print("── mantainance (was: EMPTY table) ──")
for w, s in v3.suggest("mantainance", left="track", top=5)[0]:
    print(f"   {w:<14} {s:>8.2f}")
print("\n── sucraty (truth was unreachable) ──")
for w, s in v3.suggest("sucraty", left="cyper", top=8)[0]:
    print(f"   {w:<14} {s:>8.2f}")
print("\n── the test sentence, V3 ──")
for r in v3.analyze("I hv a frind sh ws in tje univrsty lts yer"):
    if r["status"] != "ok":
        best = r["suggestions"][0][0] if r["suggestions"] else "—"
        print(f"   {r['token']:<10} {r['status']:<16} → {best}")

# ── Re-measure: strict rank-of-truth on all 4,208 pairs ──
print("\nRe-running the benchmark on V3 — ~4-6 min…")
t0 = time.time()
hit1 = hit5 = n = 0
for i, (miss, corr) in enumerate(pairs.items()):
    if i % 1000 == 0: print(f"  {i:,}/{len(pairs):,}  ({time.time()-t0:.0f}s)")
    truths = set(corr if isinstance(corr, list) else [corr])
    ranked, _ = v3.suggest(miss, top=5)
    words = [w for w, _ in ranked]
    if not words: continue
    n += 1
    hit1 += words[0] in truths
    hit5 += bool(truths & set(words))
print(f"\nV3 benchmark  ({time.time()-t0:.0f}s total):")
print(f"  accuracy@1: {hit1/len(pairs):.1%}   (v1 tiered: 80.9%)")
print(f"  accuracy@5: {hit5/len(pairs):.1%}   (v1 tiered: 96.0%)")

── mantainance (was: EMPTY table) ──
   maintainable     -16.95
   maintenance      -17.83
   montanans        -19.65
   mantinean        -20.25
   maintenances     -20.55

── sucraty (truth was unreachable) ──
   sucrate          -12.45
   scrath           -13.95
   encraty          -14.25
   eucrasy          -14.25
   sacrary          -14.25
   scrat            -14.85
   scray            -14.85
   surat            -14.85

── the test sentence, V3 ──
   frind      non_word         → find
   sh         suspicious_word  → so
   ws         suspicious_word  → as
   tje        non_word         → the
   univrsty   non_word         → university
   lts        non_word         → its
   yer        suspicious_word  → yet

Re-running the benchmark on V3 — ~4-6 min…
  0/4,304  (0s)
  1,000/4,304  (48s)
  2,000/4,304  (94s)
  3,000/4,304  (142s)
  4,000/4,304  (189s)

V3 benchmark  (201s total):
  accuracy@1: 63.1%   (v1 tiered: 80.9%)
  accuracy@5: 90.4%   (v1 tiered: 96.0%)


In [4]:
# ── Cell 3: Channels as tier 3 (append-only) + confirm metrics ────────────
class SpellCorrectorV3(corrector.SpellCorrector):
    """Phase 1 (shipped form): edit tiers rank first, exactly as v1;
    channel candidates (soundex ∪ skeleton) fill remaining top-k slots.
    v1 accuracy@1/@5 preserved by construction; channels add reach, not risk."""
    SND_LEN_WINDOW = 2
    SND_CAP        = 200

    def __init__(self, data_dir, **kw):
        super().__init__(data_dir, **kw)
        self.sound_index, self.skel_index = defaultdict(list), defaultdict(list)
        for w in self.dictionary:
            self.sound_index[soundex(w)].append(w)
            self.skel_index[skeleton(w)].append(w)

    def _channel_candidates(self, word):
        skel = set(self.skel_index.get(skeleton(word), []))
        bucket = self.sound_index.get(soundex(word), [])
        snd = sorted((w for w in bucket
                      if abs(len(w) - len(word)) <= self.SND_LEN_WINDOW),
                     key=lambda w: (abs(len(w) - len(word)), w))[:self.SND_CAP]
        return (skel | set(snd)) - {word}

    def suggest(self, word, left=None, right=None, top=5):
        if word in self.dictionary:
            return [(word, 0.0)], 0
        def rank(cands):
            return sorted(((w, -self.lam * weighted_edit_distance(word, w)
                            + self._ctx_logp_blend(w, left, right)) for w in cands),
                          key=lambda x: (-x[1], x[0]))
        c1 = self._known(self._edits1(word))
        ranked = rank(c1)
        tier = 1 if c1 else -1
        c2 = set()
        if len(ranked) < top:
            c2 = self._known({e2 for e1 in self._edits1(word)
                              for e2 in self._edits1(e1)}) - c1
            ranked += rank(c2)
            if not c1:
                tier = 2 if c2 else -1
        if len(ranked) < top:                      # tier 3: the channels
            ch = self._channel_candidates(word) - c1 - c2
            ranked += rank(ch)
            if tier == -1 and ch:
                tier = 3
        return ranked[:top], tier

v3 = SpellCorrectorV3(DATA_DIR)

print("── mantainance, top-8 (was: EMPTY) ──")
for w, s in v3.suggest("mantainance", left="track", top=8)[0]:
    print(f"   {w:<14} {s:>8.2f}")

# ── Confirm v1 metrics intact + measure the new reach at rank 10 ──
print("\nBenchmark: acc@1/@5 must equal v1; acc@10 is the channels' new metric…")
t0 = time.time()
hit1 = hit5 = hit10 = ch_rescued = 0
for i, (miss, corr) in enumerate(pairs.items()):
    if i % 1000 == 0: print(f"  {i:,}/{len(pairs):,}  ({time.time()-t0:.0f}s)")
    truths = set(corr if isinstance(corr, list) else [corr])
    ranked, _ = v3.suggest(miss, top=10)
    words = [w for w, _ in ranked]
    hit1  += bool(words) and words[0] in truths
    hit5  += bool(truths & set(words[:5]))
    hit10 += bool(truths & set(words))
    if truths & set(words) and not truths & set(words[:5]):
        ch_rescued += 1
print(f"\nV3-tiered benchmark  ({time.time()-t0:.0f}s):")
print(f"  accuracy@1:  {hit1/len(pairs):.1%}   (must be 80.9%)")
print(f"  accuracy@5:  {hit5/len(pairs):.1%}   (must be 96.0%)")
print(f"  accuracy@10: {hit10/len(pairs):.1%}  ← the channels' contribution")
print(f"  truths newly visible at ranks 6–10: {ch_rescued}")

── mantainance, top-8 (was: EMPTY) ──
   maintainable     -16.95
   maintenance      -17.83
   montanans        -19.65
   mantinean        -20.25
   maintenances     -20.55
   maintaining      -20.85
   manhattans       -21.75
   montanize        -21.75

Benchmark: acc@1/@5 must equal v1; acc@10 is the channels' new metric…
  0/4,304  (0s)
  1,000/4,304  (50s)
  2,000/4,304  (97s)
  3,000/4,304  (146s)
  4,000/4,304  (193s)

V3-tiered benchmark  (207s):
  accuracy@1:  79.5%   (must be 80.9%)
  accuracy@5:  94.1%   (must be 96.0%)
  accuracy@10: 95.4%  ← the channels' contribution
  truths newly visible at ranks 6–10: 54


In [5]:
# ── Cell 4: Canonical eval — NB05's filter, exact v1 comparability ────────
print("Applying the canonical filter (miss not in dictionary; truth reachable)…")
eval_pairs = []
for miss, corr in pairs.items():
    truths = [t for t in (corr if isinstance(corr, list) else [corr])
              if t in v3.dictionary]
    if truths and miss not in v3.dictionary:
        eval_pairs.append((miss, set(truths)))
print(f"  {len(eval_pairs):,} pairs (canonical was 4,208)")

t0 = time.time()
per_pair = []
hit1 = hit5 = hit10 = 0
for i, (miss, truths) in enumerate(eval_pairs):
    if i % 1000 == 0: print(f"  {i:,}/{len(eval_pairs):,}  ({time.time()-t0:.0f}s)")
    ranked, tier = v3.suggest(miss, top=10)
    words = [w for w, _ in ranked]
    rank = next((j + 1 for j, w in enumerate(words) if w in truths), None)
    per_pair.append({"miss": miss, "rank": rank, "tier": tier})
    hit1  += rank == 1
    hit5  += rank is not None and rank <= 5
    hit10 += rank is not None

n = len(eval_pairs)
print(f"\nV3-tiered, canonical base ({time.time()-t0:.0f}s):")
print(f"  accuracy@1:  {hit1/n:.1%}   (v1: 80.9%)")
print(f"  accuracy@5:  {hit5/n:.1%}   (v1: 96.0%)")
print(f"  accuracy@10: {hit10/n:.1%}")

# where do channel candidates specifically save the day?
t3_hits = sum(1 for p in per_pair if p["tier"] == 3 and p["rank"] == 1)
r610    = sum(1 for p in per_pair if p["rank"] and p["rank"] > 5)
print(f"  rank-1 wins on previously-EMPTY pools (tier 3): {t3_hits}")
print(f"  truths at ranks 6–10 (channel reach): {r610}")

# save the canonical eval artefact for every later phase to re-run against
artefact = {
    "engine": "V3-tiered (edit tiers + soundex/skeleton channels, trust map)",
    "n_pairs": n,
    "metrics": {"acc1": round(hit1/n, 4), "acc5": round(hit5/n, 4),
                "acc10": round(hit10/n, 4)},
    "per_pair": per_pair,
}
out = DATA_DIR / "v2_eval_baseline.json"
out.write_text(json.dumps(artefact))
print(f"\nSaved {out.name} ({out.stat().st_size/1e6:.1f} MB) — "
      "the bar every later phase must beat.")

Applying the canonical filter (miss not in dictionary; truth reachable)…
  4,208 pairs (canonical was 4,208)
  0/4,208  (0s)
  1,000/4,208  (48s)
  2,000/4,208  (96s)
  3,000/4,208  (144s)
  4,000/4,208  (190s)

V3-tiered, canonical base (198s):
  accuracy@1:  81.3%   (v1: 80.9%)
  accuracy@5:  96.3%   (v1: 96.0%)
  accuracy@10: 97.6%
  rank-1 wins on previously-EMPTY pools (tier 3): 10
  truths at ranks 6–10 (channel reach): 54

Saved v2_eval_baseline.json (0.2 MB) — the bar every later phase must beat.


In [6]:
# ── Cell 5: Export the V3 corrector and verify equivalence ────────────────
MODULE_SOURCE = '''"""
SpellCorrector — the SciSpell correction engine (V3).
Notebooks 04 (core), 08 (trust map), 09 (candidate channels) are the source
of truth. V3 = V2 + two index-based candidate channels (Soundex, consonant
skeleton) appended as tier 3, so edit-tier ranking is preserved exactly.
All indexes and trust data are rebuilt from artefacts at load — change the
dictionary or word_trust.json and the engine follows.
"""
import json
import math
from collections import defaultdict
from pathlib import Path

from edit_distance import tokenize, weighted_edit_distance
from language_model import BigramLM

ALPHABET = "abcdefghijklmnopqrstuvwxyz'"

_SOUNDEX = {**{c: "1" for c in "bfpv"}, **{c: "2" for c in "cgjkqsxz"},
            **{c: "3" for c in "dt"}, "l": "4", **{c: "5" for c in "mn"}, "r": "6"}

def soundex(w):
    """Classic Soundex: first letter + 3 digits, adjacent duplicates merged."""
    w = [c for c in w.lower() if c.isalpha()]
    if not w:
        return ""
    first = w[0]
    codes, prev = [], _SOUNDEX.get(first, "")
    for c in w[1:]:
        d = _SOUNDEX.get(c, "")
        if c in "hw":
            continue
        if d and d != prev:
            codes.append(d)
        prev = d
    return (first.upper() + "".join(codes) + "000")[:4]

VOWELS = set("aeiou")

def skeleton(w):
    """Consonant skeleton: the word with vowels removed (texting compression)."""
    s = "".join(c for c in w if c not in VOWELS)
    return s if s else w

class SpellCorrector:
    SND_LEN_WINDOW = 2
    SND_CAP = 200

    def __init__(self, data_dir, lam=3.0, m_sus=1.0):
        data_dir = Path(data_dir)
        self.lam = lam
        self.m_sus = m_sus
        self.dictionary = set(
            (data_dir / "dictionary.txt").read_text(encoding="utf-8").split("\\n"))
        self.word_freq = {w: int(c) for w, c in json.loads(
            (data_dir / "word_freq.json").read_text(encoding="utf-8")).items()}
        self.lm = BigramLM(data_dir / "language_model.json")
        conf = json.loads((data_dir / "confusion_sets.json").read_text(encoding="utf-8"))
        self.margin_sym  = conf["margins"]["symmetric"]
        self.margin_asym = conf["margins"]["asymmetric"]
        self.confusable = {w: (set(a), self.margin_sym)
                           for w, a in conf["symmetric"].items()}
        self.confusable.update({w: ({a}, self.margin_asym)
                                for w, a in conf["asymmetric"].items()})
        trust_path = data_dir / "word_trust.json"
        if trust_path.exists():
            trust = json.loads(trust_path.read_text(encoding="utf-8"))
            self.suspicious = {w: [a for a, _ in alts]
                               for w, alts in trust["suspicious"].items()}
        else:
            self.suspicious = {}
        self.sound_index, self.skel_index = defaultdict(list), defaultdict(list)
        for w in self.dictionary:
            self.sound_index[soundex(w)].append(w)
            self.skel_index[skeleton(w)].append(w)

    # ── candidate generation ──
    def _edits1(self, word):
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        return ({L + R[1:] for L, R in splits if R} |
                {L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1} |
                {L + c + R[1:] for L, R in splits if R for c in ALPHABET} |
                {L + c + R for L, R in splits for c in ALPHABET})

    def _known(self, strings):
        return {s for s in strings if s in self.dictionary}

    def _channel_candidates(self, word):
        """Tier-3 sources: sound-alikes and vowel-drop matches (index lookups)."""
        skel = set(self.skel_index.get(skeleton(word), []))
        bucket = self.sound_index.get(soundex(word), [])
        snd = sorted((w for w in bucket
                      if abs(len(w) - len(word)) <= self.SND_LEN_WINDOW),
                     key=lambda w: (abs(len(w) - len(word)), w))[:self.SND_CAP]
        return (skel | set(snd)) - {word}

    def candidates(self, word):
        if word in self.dictionary:
            return {word}, 0
        c1 = self._known(self._edits1(word))
        if c1:
            return c1, 1
        c2 = self._known({e2 for e1 in self._edits1(word) for e2 in self._edits1(e1)})
        return (c2, 2) if c2 else (set(), -1)

    # ── scoring ──
    def _ctx_logp(self, w, left, right):
        s = 0.0
        if left  is not None: s += math.log(self.lm.p_bigram(left, w))
        if right is not None: s += math.log(self.lm.p_bigram(w, right))
        if left is None and right is None: s += math.log(self.lm.p_unigram(w))
        return s

    def _ctx_logp_blend(self, w, left, right):
        s = 0.0
        if left is not None:
            s += math.log(0.7 * self.lm.p_bigram(left, w)
                          + 0.3 * self.lm.p_unigram(w))
        if right is not None:
            s += math.log(0.7 * self.lm.p_bigram(w, right)
                          + 0.3 * self.lm.p_unigram(w))
        if left is None and right is None:
            s += math.log(self.lm.p_unigram(w))
        return s

    def suggest(self, word, left=None, right=None, top=5):
        """Tiered ranking: edit tier 1, then tier 2, then channel tier 3 —
        each tier only fills capacity the previous tiers left free."""
        if word in self.dictionary:
            return [(word, 0.0)], 0
        def rank(cands):
            return sorted(((w, -self.lam * weighted_edit_distance(word, w)
                            + self._ctx_logp_blend(w, left, right)) for w in cands),
                          key=lambda x: (-x[1], x[0]))
        c1 = self._known(self._edits1(word))
        ranked = rank(c1)
        tier = 1 if c1 else -1
        c2 = set()
        if len(ranked) < top:
            c2 = self._known({e2 for e1 in self._edits1(word)
                              for e2 in self._edits1(e1)}) - c1
            ranked += rank(c2)
            if not c1:
                tier = 2 if c2 else -1
        if len(ranked) < top:
            ch = self._channel_candidates(word) - c1 - c2
            ranked += rank(ch)
            if tier == -1 and ch:
                tier = 3
        return ranked[:top], tier

    def _check_confusable(self, word, left, right):
        alts, margin = self.confusable[word]
        ranking = sorted(((w, self._ctx_logp(w, left, right))
                          for w in {word} | alts), key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        if s_best - s_word > margin:
            return True, best_alt, ranking
        return False, None, ranking

    def _check_suspicious(self, word, left, right):
        cands = [word] + self.suspicious[word]
        ranking = sorted(((w, self._ctx_logp_blend(w, left, right)) for w in cands),
                         key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        return s_best - s_word, best_alt, [(w, s) for w, s in ranking if w != word]

    # ── full pipeline ──
    def analyze(self, text, top=5):
        tokens = tokenize(text)
        results = []
        for i, tok in enumerate(tokens):
            left  = tokens[i - 1] if i > 0 else None
            right = tokens[i + 1] if i < len(tokens) - 1 else None
            if tok not in self.dictionary:
                ranked, tier = self.suggest(tok, left, right, top)
                results.append({"token": tok, "status": "non_word",
                                "suggestions": ranked, "tier": tier})
            elif tok in self.confusable:
                flagged, best, ranking = self._check_confusable(tok, left, right)
                results.append({"token": tok,
                                "status": "real_word_error" if flagged else "ok",
                                "suggestions": [(w, s) for w, s in ranking
                                                 if w != tok] if flagged else [],
                                "tier": 0})
            else:
                results.append({"token": tok, "status": "ok",
                                "suggestions": [], "tier": 0})
        if self.suspicious:
            broken = [r["status"] == "non_word" or r["token"] in self.suspicious
                      for r in results]
            for i, r in enumerate(results):
                if r["status"] != "ok" or r["token"] not in self.suspicious:
                    continue
                if not ((i > 0 and broken[i-1]) or
                        (i < len(results)-1 and broken[i+1])):
                    continue
                left  = results[i-1]["token"] if i > 0 else None
                right = results[i+1]["token"] if i < len(results)-1 else None
                gap, best, ranking = self._check_suspicious(r["token"], left, right)
                if gap > self.m_sus:
                    r.update(status="suspicious_word", suggestions=ranking[:top],
                             gap=round(gap, 1))
        return results
'''
CORRECTOR_PATH = PROJECT_ROOT / "app" / "corrector.py"
CORRECTOR_PATH.write_text(MODULE_SOURCE, encoding="utf-8")

importlib.reload(corrector)
sc3 = corrector.SpellCorrector(DATA_DIR)

# equivalence probes: module must reproduce the notebook engine exactly
import random
random.seed(42)
probe_words = (["mantainance", "sucraty", "unexepted", "frind", "tje",
                "univrsty", "lts", "recieve", "teh", "speling"]
               + random.sample([m for m, _ in eval_pairs], 30))
sent = "I hv a frind sh ws in tje univrsty lts yer"
protect = ["the sepal of the flower", "a positron is emitted",
           "shades of cyan and blue", "to allot the shares fairly",
           "the servile manner of the clerk"]

checks = [
    ("Trust map loaded", len(sc3.suspicious) == 20072, f"{len(sc3.suspicious):,}"),
    ("Channel indexes built",
     len(sc3.sound_index) == len(v3.sound_index)
     and len(sc3.skel_index) == len(v3.skel_index),
     f"{len(sc3.sound_index):,} / {len(sc3.skel_index):,}"),
    ("Module ≡ notebook on 40 suggest probes",
     all(sc3.suggest(w, top=10) == v3.suggest(w, top=10) for w in probe_words),
     "identical rankings"),
    ("mantainance top-1 present",
     bool(sc3.suggest("mantainance", top=5)[0]), "no empty tables"),
    ("Test sentence statuses",
     [r["status"] for r in sc3.analyze(sent)] ==
     [r["status"] for r in v3.analyze(sent)], "7/8 flags"),
    ("Protected sentences clean",
     all(all(r["status"] == "ok" for r in sc3.analyze(s)) for s in protect),
     f"{len(protect)} sentences"),
    ("soo → so regression",
     sc3.suggest("soo", "hem", "hey")[0][0][0] == "so", "named check"),
]
width = max(len(c[0]) for c in checks)
print("VERIFICATION\n" + "─" * (width + 30))
for label, ok, detail in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<{width}}  {detail}")
print("─" * (width + 30))
print(f"  {sum(ok for _, ok, _ in checks)}/{len(checks)} checks passed")
assert all(ok for _, ok, _ in checks)
print("\napp/corrector.py is now V3 — restart Streamlit to see it live.")

VERIFICATION
────────────────────────────────────────────────────────────────────
  PASS  Trust map loaded                        20,072
  PASS  Channel indexes built                   5,970 / 252,781
  PASS  Module ≡ notebook on 40 suggest probes  identical rankings
  PASS  mantainance top-1 present               no empty tables
  PASS  Test sentence statuses                  7/8 flags
  PASS  Protected sentences clean               5 sentences
  PASS  soo → so regression                     named check
────────────────────────────────────────────────────────────────────
  7/7 checks passed

app/corrector.py is now V3 — restart Streamlit to see it live.
